<a href="https://colab.research.google.com/github/svallejovera/cta_updated/blob/main/Fine_Tuning_a_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Classification Models

When we fine-tune a Transformer model for classification, there are a few main steps. First, we prepare labeled data and use it to train and evaluate the model. Second, once we are happy with its performance, we train a final version on the full dataset. Third, we use that final model to classify new texts.

So yes: first we teach the model, then we test whether it actually learned anything, and only then do we let it loose on the world.

### By the end of this notebook, you should be able to:

- prepare labeled text data for Transformer-based classification
- tokenize texts and choose a sensible maximum sequence length
- fine-tune a classification model with cross-validation
- interpret accuracy, weighted F1, and per-class F1
- train and save a final model together with its metadata
- use a saved model to classify new texts
- understand how the same workflow can be adapted for regression later on


### 1. Fine-Tuning a Model

We first want to train, validate, and fine-tune a model using our labeled data. We will divide the labeled data into training and validation subsets, and we will also use cross-validation to get a more stable sense of model performance.

Cross-validation helps us avoid getting too excited about a lucky split of the data. By repeating training across different folds, we get a more reliable picture of how well the model is likely to perform on new texts.

> **Note:** The first time you run a Transformer model in Colab, it may take a while because the tokenizer and model weights need to be downloaded. This is normal. The notebook is not haunted.

In [1]:
####################################################################
### NO NEED TO CHANGE ANYTHING HERE UNTIL YOU GET THE HANG OF IT ###
####################################################################

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup

import torch
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import RandomSampler, SequentialSampler

from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, classification_report

import pandas as pd
import numpy as np
import time
import datetime
import random
import os
import json

def good_update_interval(total_iters, num_desired_updates=10):
    exact_interval = total_iters / num_desired_updates
    order_of_mag = len(str(total_iters)) - 1
    round_mag = max(order_of_mag - 1, 0)
    update_interval = int(round(exact_interval, -round_mag))
    return max(update_interval, 1)

def format_time(elapsed):
    elapsed_rounded = int(round(elapsed))
    return str(datetime.timedelta(seconds=elapsed_rounded))

def print_f1_table(class_f1_dict):
    print("  ── Per-Class F1 Scores ─────────────────────────")
    for lbl, val in class_f1_dict.items():
        print(f"    {str(lbl):<20} F1: {val:.4f}")
    print("  ───────────────────────────────────────────────")

def set_seed(seed_val=6):
    random.seed(seed_val)
    np.random.seed(seed_val)
    torch.manual_seed(seed_val)
    torch.cuda.manual_seed_all(seed_val)
    torch.cuda.empty_cache()

set_seed(6)

Our labeled data in this example are news articles originally coded into five categories: politics, sport, tech, entertainment, and business. For the sake of a simple classroom example, I recoded political articles as `1` and a random sample of the rest as `0`.

The logic, however, is fully general. If your own dataset has three categories, seven categories, or even a continous variable, the same workflow still applies.

In [2]:
####################################################################
### YOU NEED TO IMPORT YOUR LABELED DATA HERE. CHANGE THIS CELL. ###
####################################################################

# Load your labeled data
data = pd.read_csv("https://raw.githubusercontent.com/svallejovera/iesp-uerj/main/politics_sample.csv")

# Define the key columns once and reuse them throughout the notebook
text_col = "Text"              # <- CHANGE THIS TO YOUR TEXT COLUMN
target_col = "Label_Politics"  # <- CHANGE THIS TO YOUR LABEL COLUMN

# Quick check
print(data[[text_col, target_col]].head())
print("\nLabel counts:")
print(data[target_col].value_counts(dropna=False))

                                                Text  Label_Politics
0  Budget to set scene for election\n \n Gordon B...               1
1  Army chiefs in regiments decision\n \n Militar...               1
2  Howard denies split over ID cards\n \n Michael...               1
3  Observers to monitor UK election\n \n Minister...               1
4  Kilroy names election seat target\n \n Ex-chat...               1

Label counts:
Label_Politics
0    420
1    417
Name: count, dtype: int64


In [11]:
####################################################################
### NO NEED TO CHANGE ANYTHING HERE UNTIL YOU GET THE HANG OF IT ###
####################################################################

# Shuffle data
data = data.sample(frac=1, random_state=6).reset_index(drop=True)

# Make sure target is plain Python int
data[target_col] = data[target_col].astype(int)

labels_unique = sorted(data[target_col].unique().tolist())

label2id = {str(label): int(i) for i, label in enumerate(labels_unique)}
id2label = {int(i): str(label) for i, label in enumerate(labels_unique)}

data["label"] = data[target_col].astype(str).map(label2id)

num_labels = len(label2id)

print("Detected labels:", labels_unique)
print(data["label"].value_counts())
print(label2id)
print(id2label)

Detected labels: [0, 1]
label
0    420
1    417
Name: count, dtype: int64
{'0': 0, '1': 1}
{0: '0', 1: '1'}


In [12]:
###########################################################################
### CHANGE ONLY THE TEXT COLUMN NAME ABOVE UNLESS YOU WANT A NEW MODEL. ###
###########################################################################

# Choose a tokenizer that matches your model
tokenizer = AutoTokenizer.from_pretrained("roberta-base", do_lower_case=True)

# If you want to try another model later, this is one place where you would change it.
# Example:
# tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

# Check sequence lengths before deciding on max_len
lengths = []

for _, row in data.iterrows():
    encoded_sent = tokenizer.encode(
        str(row[text_col]),
        add_special_tokens=True
    )
    lengths.append(len(encoded_sent))

print(f"{len(lengths):,} texts")
print(f"Min length: {min(lengths):,} tokens")
print(f"Max length: {max(lengths):,} tokens")
print(f"Median length: {int(np.median(lengths)):,} tokens")

Token indices sequence length is longer than the specified maximum sequence length for this model (1428 > 512). Running this sequence through the model will result in indexing errors


837 texts
Min length: 112 tokens
Max length: 5,390 tokens
Median length: 487 tokens


Transformer models have a maximum input length, and longer sequences require more memory. In practice, this means we often truncate texts to a fixed maximum length.

Choosing `max_len` is therefore a tradeoff: larger values preserve more information, but require more computing power. In other words, this is where theory meets the sad reality of GPU memory, geopolitics, and resource constrains.

In [13]:
#################################################################################
### YOU CAN CHANGE max_len IF YOU WANT, BUT KEEP AN EYE ON MEMORY/RUNTIME.   ###
#################################################################################

max_len = 120  # Only for the sake of this example

num_truncated = int(np.sum(np.greater(lengths, max_len)))
num_sentences = len(lengths)
prcnt = float(num_truncated) / float(num_sentences)

print(
    "{:,} of {:,} texts ({:.1%}) are longer than {} tokens.".format(
        num_truncated, num_sentences, prcnt, max_len
    )
)

836 of 837 texts (99.9%) are longer than 120 tokens.


In [14]:
####################################################################
### TOKENIZE THE FULL DATASET AND BUILD A TensorDataset OBJECT.   ###
####################################################################

labels = []
input_ids = []
attn_masks = []

for _, row in data.iterrows():
    encoded_dict = tokenizer(
        str(row[text_col]),
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    input_ids.append(encoded_dict["input_ids"])
    attn_masks.append(encoded_dict["attention_mask"])
    labels.append(row["label"])

input_ids = torch.cat(input_ids, dim=0)
attn_masks = torch.cat(attn_masks, dim=0)
labels = torch.tensor(labels, dtype=torch.long)

dataset = TensorDataset(input_ids, attn_masks, labels)

print("TensorDataset ready.")
print(f"input_ids shape: {input_ids.shape}")
print(f"attention masks shape: {attn_masks.shape}")
print(f"labels shape: {labels.shape}")

TensorDataset ready.
input_ids shape: torch.Size([837, 120])
attention masks shape: torch.Size([837, 120])
labels shape: torch.Size([837])


> What are tensors? Tensors organize data in multidimensional arrays. Tensor methods organize neural network weights in a "data tensor", analyze and reduce the number of neural network weights. It helps with the computation of high-dimension matrices (e.g., word embeddings), efficiently parallelized using CUDA (NVidia), of using TPUs (Tensor Processing Units) developed by Google.

In [15]:
################################################################################
### YOU CAN PLAY WITH THESE PARAMETERS LATER.                                ###
################################################################################

model_name = "roberta-base"   # must match the tokenizer family
lr = 2e-5 # between 2e-6 and 2e-5
epochs = 2 # keep an eye on overfitting, but usually 5 is already too much
batch_size = 8 # better if increased by 2^n
num_labels = len(labels_unique)

print({
    "model_name": model_name,
    "lr": lr,
    "epochs": epochs,
    "batch_size": batch_size,
    "num_labels": num_labels
})

{'model_name': 'roberta-base', 'lr': 2e-05, 'epochs': 2, 'batch_size': 8, 'num_labels': 2}


In [16]:
####################################################################
### NO NEED TO CHANGE ANYTHING HERE UNTIL YOU GET THE HANG OF IT ###
####################################################################

set_seed(6)

fold_stats = []
total_t0 = time.time()

To determine which hyperparameters to use and to evaluate model performance, we use cross-validation (CV). The data are divided into $k$ folds. In each run, one fold is held out for validation while the remaining $k-1$ folds are used for training. This process repeats until every fold has served as the validation fold once.

The main idea is simple: performance should not depend too much on one lucky or unlucky split of the data.

For this example, we will use only two folds to keep the runtime manageable in class. In real work, you would often use more (ten is common).

In [17]:
###########################################################################
### CHANGE ONLY k_folds IF YOU WANT TO RUN MORE OR FEWER CV ITERATIONS. ###
###########################################################################

k_folds = 2
kfold = KFold(n_splits=k_folds, shuffle=True, random_state=6)

timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H%M")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for fold, (train_ids, test_ids) in enumerate(kfold.split(dataset)):
    print(f"\n======== Fold {fold + 1} / {k_folds} ========")

    train_subsampler = torch.utils.data.SubsetRandomSampler(train_ids)
    test_subsampler = torch.utils.data.SubsetRandomSampler(test_ids)

    train_dataloader = DataLoader(dataset, batch_size=batch_size, sampler=train_subsampler)
    test_dataloader = DataLoader(dataset, batch_size=batch_size, sampler=test_subsampler)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id
    )
    model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, eps=1e-6)
    total_steps = len(train_dataloader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=10,
        num_training_steps=total_steps
    )

    best_val_loss = float("inf")
    patience = 2
    patience_counter = 0

    for epoch_i in range(epochs):
        print(f"\nEpoch {epoch_i + 1}/{epochs}")
        model.train()
        total_train_loss = 0
        t0 = time.time()

        update_interval = good_update_interval(len(train_dataloader), 10)

        for step, batch in enumerate(train_dataloader):
            if step % update_interval == 0 and step != 0:
                print(f"  Batch {step} of {len(train_dataloader)} ...")

            b_input_ids, b_input_mask, b_labels = [x.to(device) for x in batch]
            model.zero_grad()

            outputs = model(
                b_input_ids,
                attention_mask=b_input_mask,
                labels=b_labels
            )
            loss = outputs.loss
            total_train_loss += loss.item()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        avg_train_loss = total_train_loss / len(train_dataloader)
        training_time = format_time(time.time() - t0)

        model.eval()
        total_eval_loss = 0
        preds, trues = [], []

        for batch in test_dataloader:
            b_input_ids, b_input_mask, b_labels = [x.to(device) for x in batch]

            with torch.no_grad():
                outputs = model(
                    b_input_ids,
                    attention_mask=b_input_mask,
                    labels=b_labels
                )

            loss = outputs.loss
            logits = outputs.logits

            total_eval_loss += loss.item()
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            trues.extend(b_labels.cpu().numpy())

        avg_val_loss = total_eval_loss / len(test_dataloader)
        acc = accuracy_score(trues, preds)
        f1_weighted = f1_score(trues, preds, average="weighted")

        per_class_f1 = f1_score(
            trues,
            preds,
            average=None,
            labels=list(range(num_labels))
        )
        class_f1_dict = {id2label[i]: float(per_class_f1[i]) for i in range(num_labels)}

        print(f"  Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        print(f"  Accuracy: {acc:.4f} | Weighted F1: {f1_weighted:.4f}")
        print_f1_table(class_f1_dict)
        print(f"  Time: {training_time}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            print(f"  Validation loss did not improve. Patience {patience_counter}/{patience}.")

        fold_record = {
            "fold": fold + 1,
            "Train Loss": avg_train_loss,
            "Val Loss": avg_val_loss,
            "Accuracy": acc,
            "Weighted F1": f1_weighted
        }
        fold_record.update({f"F1_{lbl}": val for lbl, val in class_f1_dict.items()})
        fold_stats.append(fold_record)


======== Fold 1 / 2 ========


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Epoch 1/2
  Batch 5 of 53 ...
  Batch 10 of 53 ...
  Batch 15 of 53 ...
  Batch 20 of 53 ...
  Batch 25 of 53 ...
  Batch 30 of 53 ...
  Batch 35 of 53 ...
  Batch 40 of 53 ...
  Batch 45 of 53 ...
  Batch 50 of 53 ...
  Train Loss: 0.4638 | Val Loss: 0.0577
  Accuracy: 0.9833 | Weighted F1: 0.9833
  ── Per-Class F1 Scores ─────────────────────────
    0                    F1: 0.9832
    1                    F1: 0.9834
  ───────────────────────────────────────────────
  Time: 0:00:11

Epoch 2/2
  Batch 5 of 53 ...
  Batch 10 of 53 ...
  Batch 15 of 53 ...
  Batch 20 of 53 ...
  Batch 25 of 53 ...
  Batch 30 of 53 ...
  Batch 35 of 53 ...
  Batch 40 of 53 ...
  Batch 45 of 53 ...
  Batch 50 of 53 ...
  Train Loss: 0.0319 | Val Loss: 0.0446
  Accuracy: 0.9881 | Weighted F1: 0.9881
  ── Per-Class F1 Scores ─────────────────────────
    0                    F1: 0.9880
    1                    F1: 0.9881
  ───────────────────────────────────────────────
  Time: 0:00:10

======== Fold 2 / 2

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Epoch 1/2
  Batch 5 of 53 ...
  Batch 10 of 53 ...
  Batch 15 of 53 ...
  Batch 20 of 53 ...
  Batch 25 of 53 ...
  Batch 30 of 53 ...
  Batch 35 of 53 ...
  Batch 40 of 53 ...
  Batch 45 of 53 ...
  Batch 50 of 53 ...
  Train Loss: 0.5127 | Val Loss: 0.0775
  Accuracy: 0.9809 | Weighted F1: 0.9809
  ── Per-Class F1 Scores ─────────────────────────
    0                    F1: 0.9810
    1                    F1: 0.9808
  ───────────────────────────────────────────────
  Time: 0:00:10

Epoch 2/2
  Batch 5 of 53 ...
  Batch 10 of 53 ...
  Batch 15 of 53 ...
  Batch 20 of 53 ...
  Batch 25 of 53 ...
  Batch 30 of 53 ...
  Batch 35 of 53 ...
  Batch 40 of 53 ...
  Batch 45 of 53 ...
  Batch 50 of 53 ...
  Train Loss: 0.0419 | Val Loss: 0.0430
  Accuracy: 0.9880 | Weighted F1: 0.9880
  ── Per-Class F1 Scores ─────────────────────────
    0                    F1: 0.9883
    1                    F1: 0.9878
  ───────────────────────────────────────────────
  Time: 0:00:10


### How to read the loss values

- **Validation loss much larger than training loss** usually suggests overfitting.
- **Validation loss slightly larger than training loss** is common and often fine.
- **Training and validation loss both high** usually suggests underfitting or weak settings.
- **The main goal** is not to get the training loss to look heroic; it is to get good validation performance.

Also, for imbalanced classification tasks, weighted F1 is often more informative than accuracy alone. You can check the meaning of each stat, how to interpret them, and when to use them, in scholarly journal [Wikipedia](https://en.wikipedia.org/wiki/Precision_and_recall), or check out [this paper](https://d1wqtxts1xzle7.cloudfront.net/37219940/5215ijdkp01-libre.pdf?1428316763=&response-content-disposition=inline%3B+filename%3DA_REVIEW_ON_EVALUATION_METRICS_FOR_DATA.pdf&Expires=1709137264&Signature=f3EFHnlTZXa38ug6~VBumSZrfe9ECAyMUh04CNTzYnXEsVaJS3T12eNPbu7iNP~z3DSTTJ2NAV845v50XBe8Sjm7AylacfjGxcQ8YqaDsMulhkCV8c-JtTrWaLILlSUzbQp9M5Md3ubChx5Y9xkBp~s~XlecEEu9B5QEOjyr2aiZRA6gz98crSv0VKKV2ow986UxoSaWZgaYPmTsTrWU2EN3-0S1~OyO9tf2eFqbb3jUwOl15vX1rzzoG9lcpqbURB0eGMqPlXoWPHYBAlGmvUJOGxfkz15VpCxYtg-RoL5IYJONHlkV8GDWXntOm4WdY-ZIcgF3f3c7XhpDzgzvGw__&Key-Pair-Id=APKAJLOHF5GGSLRBV4ZA).


We can now inspect the fold-level output and summarize performance across folds. This is where we stop admiring the model's confidence and start asking whether it actually deserves that confidence.

In [18]:
folds_df = pd.DataFrame(fold_stats)
folds_df

,fold,Train Loss,Val Loss,Accuracy,Weighted F1,F1_0,F1_1
0,1,0.463769,0.057717,0.983294,0.983294,0.983213,0.983373
1,1,0.031918,0.044632,0.988067,0.988067,0.988010,0.988124
2,2,0.512749,0.077466,0.980861,0.980863,0.980952,0.980769
3,2,0.041931,0.042967,0.988038,0.988038,0.988290,0.987775


In [19]:
results_summary = {
    "Model": model_name,
    "LR": lr,
    "Epochs": epochs,
    "Batch Size": batch_size,
    "Mean Acc": float(np.mean([f["Accuracy"] for f in fold_stats])),
    "SD Acc": float(np.std([f["Accuracy"] for f in fold_stats])),
    "Mean Weighted F1": float(np.mean([f["Weighted F1"] for f in fold_stats])),
    "SD Weighted F1": float(np.std([f["Weighted F1"] for f in fold_stats]))
}

for lbl in labels_unique:
    f1_vals = [f[f"F1_{lbl}"] for f in fold_stats if f"F1_{lbl}" in f]
    results_summary[f"Mean F1_{lbl}"] = float(np.mean(f1_vals))
    results_summary[f"SD F1_{lbl}"] = float(np.std(f1_vals))

summary_df = pd.DataFrame([results_summary])
summary_df

,Model,LR,Epochs,Batch Size,Mean Acc,SD Acc,Mean Weighted F1,SD Weighted F1,Mean F1_0,SD F1_0,Mean F1_1,SD F1_1
0,roberta-base,0.00002,2,8,0.985065,0.003109,0.985065,0.003108,0.985116,0.003139,0.98501,0.003082


In [20]:
print("Per-fold records:")
print(folds_df)

print("\nAggregate summary:")
print(summary_df)

Per-fold records:
   fold  Train Loss  Val Loss  Accuracy  Weighted F1      F1_0      F1_1
0     1    0.463769  0.057717  0.983294     0.983294  0.983213  0.983373
1     1    0.031918  0.044632  0.988067     0.988067  0.988010  0.988124
2     2    0.512749  0.077466  0.980861     0.980863  0.980952  0.980769
3     2    0.041931  0.042967  0.988038     0.988038  0.988290  0.987775

Aggregate summary:
          Model       LR  Epochs  Batch Size  Mean Acc    SD Acc  \
0  roberta-base  0.00002       2           8  0.985065  0.003109   

   Mean Weighted F1  SD Weighted F1  Mean F1_0   SD F1_0  Mean F1_1   SD F1_1  
0          0.985065        0.003108   0.985116  0.003139    0.98501  0.003082  


All right, now that the model seems to know what it is doing, we can train a final version using the labeled dataset and save it for later use.

Instead of training blindly on everything and hoping for the best, we will still keep a small validation split during this final training step. That gives us one more check before saving the model.


In [21]:
#######################################################################
### FINAL TRAINING: TRAIN/VALIDATION SPLIT + MODEL/METADATA SAVING. ###
#######################################################################

# Train/validation split
train_inputs, val_inputs, train_masks, val_masks, train_labels, val_labels = train_test_split(
    input_ids,
    attn_masks,
    labels,
    test_size=0.10,
    random_state=6,
    stratify=labels
)

train_dataset = TensorDataset(train_inputs, train_masks, train_labels)
val_dataset   = TensorDataset(val_inputs, val_masks, val_labels)

train_dataloader = DataLoader(
    train_dataset,
    sampler=RandomSampler(train_dataset),
    batch_size=batch_size
)

validation_dataloader = DataLoader(
    val_dataset,
    sampler=SequentialSampler(val_dataset),
    batch_size=batch_size
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, eps=1e-6)
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=10,
    num_training_steps=total_steps
)

total_t0 = time.time()

for epoch_i in range(epochs):
    print(f"\n======== Epoch {epoch_i + 1} / {epochs} ========")

    model.train()
    total_train_loss = 0
    t0 = time.time()

    update_interval = good_update_interval(len(train_dataloader), 10)

    for step, batch in enumerate(train_dataloader):
        if step % update_interval == 0 and step != 0:
            print(f"  Batch {step} of {len(train_dataloader)} ...")

        b_input_ids, b_input_mask, b_labels = [x.to(device) for x in batch]

        model.zero_grad()

        outputs = model(
            b_input_ids,
            attention_mask=b_input_mask,
            labels=b_labels
        )
        loss = outputs.loss
        total_train_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

    avg_train_loss = total_train_loss / len(train_dataloader)
    training_time = format_time(time.time() - t0)

    print(f"  Average training loss: {avg_train_loss:.4f}")
    print(f"  Training epoch took: {training_time}")

    model.eval()
    total_eval_loss = 0
    preds, trues = [], []

    for batch in validation_dataloader:
        b_input_ids, b_input_mask, b_labels = [x.to(device) for x in batch]

        with torch.no_grad():
            outputs = model(
                b_input_ids,
                attention_mask=b_input_mask,
                labels=b_labels
            )

        loss = outputs.loss
        logits = outputs.logits

        total_eval_loss += loss.item()
        preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        trues.extend(b_labels.cpu().numpy())

    avg_val_loss = total_eval_loss / len(validation_dataloader)
    acc = accuracy_score(trues, preds)
    f1_weighted = f1_score(trues, preds, average="weighted")

    print(f"  Validation Loss: {avg_val_loss:.4f}")
    print(f"  Validation Accuracy: {acc:.4f}")
    print(f"  Validation Weighted F1: {f1_weighted:.4f}")

total_training_time = format_time(time.time() - total_t0)
print(f"\nTraining complete in {total_training_time}.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



======== Epoch 1 / 2 ========
  Batch 10 of 95 ...
  Batch 20 of 95 ...
  Batch 30 of 95 ...
  Batch 40 of 95 ...
  Batch 50 of 95 ...
  Batch 60 of 95 ...
  Batch 70 of 95 ...
  Batch 80 of 95 ...
  Batch 90 of 95 ...
  Average training loss: 0.2849
  Training epoch took: 0:00:19
  Validation Loss: 0.2011
  Validation Accuracy: 0.9643
  Validation Weighted F1: 0.9642

======== Epoch 2 / 2 ========
  Batch 10 of 95 ...
  Batch 20 of 95 ...
  Batch 30 of 95 ...
  Batch 40 of 95 ...
  Batch 50 of 95 ...
  Batch 60 of 95 ...
  Batch 70 of 95 ...
  Batch 80 of 95 ...
  Batch 90 of 95 ...
  Average training loss: 0.0198
  Training epoch took: 0:00:20
  Validation Loss: 0.0140
  Validation Accuracy: 0.9881
  Validation Weighted F1: 0.9881

Training complete in 0:00:40.


In [22]:
####################################################################
### SAVE MODEL, TOKENIZER, AND LABEL MAPS FOR FUTURE PREDICTION. ###
####################################################################

save_dir = "my_fine_tuned_classifier"   # <- CHANGE THIS FOLDER NAME IF YOU WANT
os.makedirs(save_dir, exist_ok=True)

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

meta = {
    "label2id": label2id,
    "id2label": id2label,
    "model_name": model_name,
    "max_len": max_len,
    "lr": lr,
    "epochs": epochs,
    "batch_size": batch_size,
    "num_labels": num_labels,
    "text_col": text_col,
    "target_col": target_col
}

meta_row = meta.copy()
meta_row["label2id"] = json.dumps(meta_row["label2id"], ensure_ascii=False)
meta_row["id2label"] = json.dumps(meta_row["id2label"], ensure_ascii=False)

meta_df = pd.DataFrame([meta_row])
meta_csv = os.path.join(save_dir, "model_meta.csv")
meta_df.to_csv(meta_csv, index=False)

print(f"Model saved to: {save_dir}")
print(f"Metadata saved to: {meta_csv}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: my_fine_tuned_classifier
Metadata saved to: my_fine_tuned_classifier/model_meta.csv


Why save `label2id` and `id2label`?

Because once the model is trained, you still need to know which output index corresponds to which class (labels are changed to numeric ids). Saving those mappings makes your model reusable and reproducible. Without them, you have an output `2`, but you will not be sure what `2` means.

### 3. Using the Model

To use the model later, we load the saved model, tokenizer, and metadata. Then we can classify new text.

Below is a student-friendly version that returns the predicted label and a confidence score. Confidence is not the same thing as correctness, but it is still useful information.

In [23]:
# If this were a new session, this is how you would load everything back in.

from transformers import AutoTokenizer, AutoModelForSequenceClassification

load_dir = save_dir
meta_df = pd.read_csv(os.path.join(load_dir, "model_meta.csv"))
meta = meta_df.iloc[0].to_dict()

label2id_loaded = json.loads(meta["label2id"])
id2label_loaded = json.loads(meta["id2label"])

label2id_loaded = {k: int(v) for k, v in label2id_loaded.items()}
id2label_loaded = {int(k): v for k, v in id2label_loaded.items()}

max_len_loaded = int(meta["max_len"])
num_labels_loaded = int(meta["num_labels"])

tokenizer_loaded = AutoTokenizer.from_pretrained(load_dir)
my_model = AutoModelForSequenceClassification.from_pretrained(
    load_dir,
    num_labels=num_labels_loaded,
    id2label=id2label_loaded,
    label2id=label2id_loaded
)

inference_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
my_model.to(inference_device)
my_model.eval()

def predict(text, model=my_model, tokenizer=tokenizer_loaded, id2label=id2label_loaded, max_len=max_len_loaded):
    enc = tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_len
    )

    input_ids = enc["input_ids"].to(inference_device)
    attention_mask = enc["attention_mask"].to(inference_device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    probs = torch.softmax(outputs.logits, dim=-1)
    pred_id = torch.argmax(probs, dim=-1).item()
    confidence = torch.max(probs, dim=-1).values.item()

    return {
        "predicted_id": pred_id,
        "predicted_label": id2label[pred_id],
        "confidence": confidence
    }

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [26]:
predict(
    "The NDP and the Liberals signed a confidence and supply agreement."
)

{'predicted_id': 1, 'predicted_label': '1', 'confidence': 0.9708074927330017}

In [25]:
predict(
    "Footy Headlines can leak the Real Madrid 24-25 home kit, as worn by Bellingham. It has a no-nonsense design in white and black."
)

{'predicted_id': 0, 'predicted_label': '0', 'confidence': 0.9993959665298462}

And here is some code to loop through a corpus more neatly so you can pass a larger dataset through your fine-tuned model.


In [27]:
from tqdm import tqdm

# Example target corpus
df = pd.read_parquet(
    "hf://datasets/Themira/en_si_news_classification_with_label_name/data/train_en-00000-of-00001.parquet"
)
df_sample = df.head(9).copy()
print(df_sample.head())

                                            sentence          label
0  Russian Skating Star Is 'Lighthearted' At Prac...         Sports
1  Pete Buttigieg Rejects Notion That Black Voter...      Political
2  Jay Z Is Making A Movie And Docuseries Based O...  Entertainment
3  See All The Looks From The 2018 Golden Globes ...  Entertainment
4  Jimmy Fallon Calls Out Mystery Object Coming F...  Entertainment


In [30]:
predictions = []

for text in tqdm(df_sample["sentence"]):
    pred_temp = predict(text)
    predictions.append(pred_temp)

pred_df = pd.DataFrame(predictions)
df_sample = pd.concat([df_sample.reset_index(drop=True), pred_df], axis=1)

df_sample.head(n=10)
# df_sample.to_csv("pred_df_sample.csv", index=False)

100%|██████████| 9/9 [00:00<00:00, 96.83it/s]


,sentence,label,predicted_id,predicted_label,confidence,predicted_id,predicted_label,confidence,predicted_id,predicted_label,confidence
0,Russian Skating Star Is 'Lighthearted' At Prac...,Sports,0,0,0.999404,0,0,0.999404,0,0,0.999404
1,Pete Buttigieg Rejects Notion That Black Voter...,Political,0,0,0.995572,0,0,0.995572,0,0,0.995572
2,Jay Z Is Making A Movie And Docuseries Based O...,Entertainment,0,0,0.999217,0,0,0.999217,0,0,0.999217
3,See All The Looks From The 2018 Golden Globes ...,Entertainment,0,0,0.999139,0,0,0.999139,0,0,0.999139
4,Jimmy Fallon Calls Out Mystery Object Coming F...,Entertainment,0,0,0.999097,0,0,0.999097,0,0,0.999097
5,Samantha Bee Gets Candid About Dealing With Tw...,Entertainment,0,0,0.998788,0,0,0.998788,0,0,0.998788
6,Exes Bella Hadid And The Weeknd Spotted Kissin...,Entertainment,0,0,0.999178,0,0,0.999178,0,0,0.999178
7,LeBron James Says Orlando Shooting Puts Import...,Sports,0,0,0.999205,0,0,0.999205,0,0,0.999205
8,House Panel Calls New Postmaster General To Ex...,Political,0,0,0.768941,0,0,0.768941,0,0,0.768941


### Optional extension: batched prediction with uncertainty

In real projects, you will often want a more scalable prediction workflow. A more advanced version can:
- load the model metadata automatically
- run batched predictions with a `DataLoader`
- save prediction confidence
- calculate normalized entropy as a rough measure of uncertainty

That is the cleaner workflow I use in my standalone scripts. The simple version above is easier for class; the batched version is better once you start running larger corpora.


### Optional extension: adapting the workflow for regression

Everything above focused on classification, where the outcome is a category. But the same general workflow can also be adapted for **regression**, where the outcome is a continuous value, such as an ideology score.

The main changes are:

- the target variable is numeric and continuous
- `num_labels = 1`
- labels are stored as floats, not integer class IDs
- training uses a regression loss such as mean squared error (MSE)
- performance is evaluated with metrics such as **MSE** and **R²**

The code cell below is provided for future reference. It is **not** meant to be run as part of this notebook.

In [ ]:
# ==============================================================
# OPTIONAL TEMPLATE: REGRESSION WITH A TRANSFORMER (DO NOT RUN)
# ==============================================================

'''
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
from torch.utils.data import TensorDataset, DataLoader
import torch
import pandas as pd
import numpy as np
import random, time, datetime

# Example setup
data = pd.read_excel("your_regression_data.xlsx")
text_col = "text_clean"
target_col = "continuous_outcome"

tokenizer = AutoTokenizer.from_pretrained("roberta-large", do_lower_case=True)
max_len = 150

input_ids, attn_masks, labels = [], [], []

for _, row in data.iterrows():
    encoded_dict = tokenizer.encode_plus(
        str(row[text_col]),
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    input_ids.append(encoded_dict["input_ids"])
    attn_masks.append(encoded_dict["attention_mask"])
    labels.append(row[target_col])

input_ids = torch.cat(input_ids, dim=0)
attn_masks = torch.cat(attn_masks, dim=0)
labels = torch.tensor(labels, dtype=torch.float).unsqueeze(1)

dataset = TensorDataset(input_ids, attn_masks, labels)

model_name = "roberta-large"
lr = 2e-5
epochs = 4
batch_size = 16

kfold = KFold(n_splits=5, shuffle=True, random_state=6)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for fold, (train_ids, test_ids) in enumerate(kfold.split(dataset)):
    train_subsampler = torch.utils.data.SubsetRandomSampler(train_ids)
    test_subsampler = torch.utils.data.SubsetRandomSampler(test_ids)

    train_dataloader = DataLoader(dataset, batch_size=batch_size, sampler=train_subsampler)
    test_dataloader = DataLoader(dataset, batch_size=batch_size, sampler=test_subsampler)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)
    model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, eps=1e-6)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=10,
        num_training_steps=len(train_dataloader) * epochs
    )
    loss_fn = torch.nn.MSELoss()

    for epoch_i in range(epochs):
        model.train()
        total_train_loss = 0

        for batch in train_dataloader:
            b_input_ids, b_input_mask, b_labels = [x.to(device) for x in batch]
            model.zero_grad()

            outputs = model(b_input_ids, attention_mask=b_input_mask)
            logits = outputs.logits

            loss = loss_fn(logits, b_labels)
            total_train_loss += loss.item()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        model.eval()
        preds, trues = [], []

        for batch in test_dataloader:
            b_input_ids, b_input_mask, b_labels = [x.to(device) for x in batch]

            with torch.no_grad():
                outputs = model(b_input_ids, attention_mask=b_input_mask)
                logits = outputs.logits

            preds.extend(logits.detach().cpu().numpy())
            trues.extend(b_labels.detach().cpu().numpy())

        preds = np.array(preds).flatten()
        trues = np.array(trues).flatten()

        mse = mean_squared_error(trues, preds)
        r2 = r2_score(trues, preds)

        print(f"Fold {fold + 1}, Epoch {epoch_i + 1}: MSE = {mse:.4f}, R² = {r2:.4f}")
'''

Cool. Now you can train a model, save it properly, and run it on a larger corpus.

Is this the last step? **No.** You still want one more layer of validation: **out-of-sample validation**. That means drawing a fresh sample from your target corpus, labeling it by hand, classifying it with your model, and comparing the predictions to the human labels.

This is also a good moment to inspect the confusion matrix. It helps you see not just whether the model makes mistakes, but what kinds of mistakes it prefers to make.